# 10 Results Aggregation (LUNAR-style)

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px

RESULTS_DIR = Path("../results")
AGG_DIR = RESULTS_DIR / "aggregation"
AGG_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(RESULTS_DIR.glob("*.json"))
records = []
for path in json_files:
    try:
        with open(path, "r") as f:
            records.append(json.load(f))
    except Exception as e:
        print("SKIP", path.name, e)

df = pd.DataFrame(records)
if df.empty:
    raise ValueError("No JSON result files found in ../results")

df.to_csv(AGG_DIR / "all_results_flat.csv", index=False)
metric_cols = [c for c in ["AUC_ROC", "AUC_PR", "Precision", "Recall", "F1", "runtime_train", "runtime_inference"] if c in df.columns]
summary = df.groupby(["dataset_name", "model_type", "fusion_strategy"], dropna=False)[metric_cols].agg(["mean", "std", "min", "max", "count"])
summary.to_csv(AGG_DIR / "summary_by_dataset_model_fusion.csv")

for metric in [m for m in ["AUC_ROC", "AUC_PR", "F1", "runtime_train"] if m in df.columns]:
    fig = px.box(df, x="model_type", y=metric, color="dataset_name", points="all", title=f"{metric} by model_type")
    fig.write_html(AGG_DIR / f"box_{metric}_by_model.html")

if "fusion_strategy" in df.columns:
    fig = px.box(df.dropna(subset=["fusion_strategy"]), x="fusion_strategy", y="F1", color="dataset_name", points="all", title="F1 by fusion strategy")
    fig.write_html(AGG_DIR / "box_f1_by_fusion.html")

df